# Re download data

In [1]:
pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!jupyter nbextension enable --py widgetsnbextension --sys-prefix

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: kernel kernelspec migrate run troubleshoot

Jupyter command `jupyter-nbextension` not found.


In [1]:
import os
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# -------------------------------
# Config
# -------------------------------
DATA_DIR = "C:/Users/CSIS-PostGrad/multimodal_pipeline/Data"
OUT_DIR = "C:/Users/CSIS-PostGrad/multimodal_pipeline/Data2"
TSV_FILES = {
    "train": os.path.join(DATA_DIR, "multimodal_train.tsv"),
    "validate": os.path.join(DATA_DIR, "multimodal_validate.tsv"),
    "test": os.path.join(DATA_DIR, "multimodal_test_public.tsv")
}
OUTPUT_DIRS = {
    "train": os.path.join(OUT_DIR, "train_images"),
    "validate": os.path.join(OUT_DIR, "validate_images"),
    "test": os.path.join(OUT_DIR, "test_images")
}
FAILED_LOG_DIR = os.path.join(OUT_DIR, "failed_downloads")
CHECKPOINT_DIR = os.path.join(OUT_DIR, "checkpoints")

os.makedirs(FAILED_LOG_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
for path in OUTPUT_DIRS.values():
    os.makedirs(path, exist_ok=True)

MAX_THREADS = 8
MAX_RETRIES = 3
BATCH_SIZE = 5000
CHECKPOINT_EVERY = 10  # batches
SESSION_TIMEOUT = 10

# -------------------------------
# Internet Connectivity
# -------------------------------
def internet_on(timeout=5):
    test_urls = ["https://www.google.com", "https://pypi.org", "https://huggingface.co"]
    for url in test_urls:
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200:
                return True
        except Exception:
            continue
    return False

def wait_for_internet():
    while not internet_on():
        print("🌐 No internet connection. Waiting 10 seconds...")
        time.sleep(10)

# -------------------------------
# Single Image Download
# -------------------------------
def download_image_with_session(row, output_dir, session):
    img_id = str(row["id"])
    url = row["image_url"]
    save_path = os.path.join(output_dir, f"{img_id}.jpg")

    if os.path.exists(save_path) or not isinstance(url, str) or not url.startswith("http"):
        return "skipped", None

    wait_for_internet()
    try:
        with session.get(url, timeout=SESSION_TIMEOUT) as resp:
            if resp.status_code == 200:
                img = Image.open(BytesIO(resp.content)).convert("RGB")
                img.save(save_path)
                return "downloaded", None
            else:
                return "failed", row
    except Exception:
        return "failed", row

# -------------------------------
# Batch Downloader with Checkpointing
# -------------------------------
def download_images_parallel(df, output_dir, split_name):
    failed_records = []
    skipped = 0
    downloaded = 0
    total = len(df)
    session = requests.Session()

    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{split_name}_checkpoint.csv")
    log_path = os.path.join(CHECKPOINT_DIR, f"{split_name}_progress_log.csv")

    # Resume from checkpoint
    completed_ids = set()
    if os.path.exists(checkpoint_path):
        print(f"🔁 Resuming from checkpoint: {checkpoint_path}")
        completed_df = pd.read_csv(checkpoint_path)
        completed_ids = set(completed_df["id"].astype(str))
        df = df[~df["id"].astype(str).isin(completed_ids)]

    batch_count = 0
    cumulative_downloaded = len(completed_ids)

    for start in range(0, len(df), BATCH_SIZE):
        batch_start_time = time.time()
        batch = df.iloc[start:start + BATCH_SIZE]
        batch_count += 1

        with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
            futures = {
                executor.submit(download_image_with_session, row, output_dir, session): row for _, row in batch.iterrows()
            }

            for future in tqdm(as_completed(futures), total=len(batch), desc=f"{split_name} {start}-{start+len(batch)}"):
                try:
                    status, fail = future.result(timeout=30)
                    if status == "downloaded":
                        downloaded += 1
                        cumulative_downloaded += 1
                    elif status == "skipped":
                        skipped += 1
                    elif status == "failed":
                        failed_records.append(fail)
                except Exception:
                    continue

        duration = round(time.time() - batch_start_time, 2)
        print(f"✅ Batch {batch_count}: Downloaded={downloaded}, Skipped={skipped}, Failed={len(failed_records)}, Time={duration}s")

        # --- Save checkpoint every N batches ---
        if batch_count % CHECKPOINT_EVERY == 0 or start + BATCH_SIZE >= total:
            checkpoint_df = pd.DataFrame({"id": df.iloc[:start + len(batch)]["id"].astype(str)})
            checkpoint_df.to_csv(checkpoint_path, index=False)

            log_entry = {
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "batch_number": batch_count,
                "images_attempted": len(batch),
                "downloaded_batch": downloaded,
                "skipped_batch": skipped,
                "failed_batch": len(failed_records),
                "cumulative_downloaded": cumulative_downloaded,
                "duration_seconds": duration
            }

            log_df = pd.DataFrame([log_entry])
            if not os.path.exists(log_path):
                log_df.to_csv(log_path, index=False)
            else:
                log_df.to_csv(log_path, mode="a", header=False, index=False)

            print(f"💾 Checkpoint + Log saved at {datetime.now().strftime('%H:%M:%S')}")

        time.sleep(1)

    print(f"✅ Final checkpoint saved: {checkpoint_path}")
    return downloaded, skipped, failed_records

# -------------------------------
# Retry logic
# -------------------------------
def retry_failed_downloads(failed_records, output_dir, split_name):
    for attempt in range(1, MAX_RETRIES + 1):
        if not failed_records:
            break
        print(f"\n🔁 Retry {attempt} for {len(failed_records)} failed items...")
        df_failed = pd.DataFrame(failed_records)
        _, _, failed_records = download_images_parallel(df_failed, output_dir, split_name)
    return failed_records

# -------------------------------
# Main Download Function
# -------------------------------
def download_split(split):
    df = pd.read_csv(TSV_FILES[split], sep="\t")
    output_dir = OUTPUT_DIRS[split]
    print(f"\n📦 Processing {split.upper()} split ({len(df)} samples)...")

    existing = {os.path.splitext(f)[0] for f in os.listdir(output_dir) if f.lower().endswith(".jpg")}
    df = df[~df["id"].astype(str).isin(existing)]
    print(f"Remaining to download: {len(df)}")

    if len(df) == 0:
        print(f"✅ {split.upper()} images already downloaded.")
        return

    downloaded, skipped, failed = download_images_parallel(df, output_dir, split)
    print(f"\n✅ Done with {split}: Downloaded={downloaded}, Skipped={skipped}, Failed={len(failed)}")

    if failed:
        retry_failed_downloads(failed, output_dir, split)
        pd.DataFrame(failed).to_csv(os.path.join(FAILED_LOG_DIR, f"failed_{split}.csv"), index=False)
        print(f"❌ Failed list saved: failed_{split}.csv")

# -------------------------------
# Run All Splits
# -------------------------------
for split in ["train", "validate", "test"]:
    download_split(split)



📦 Processing TRAIN split (564000 samples)...
Remaining to download: 563456


train 0-5000:   0%|          | 0/5000 [00:00<?, ?it/s]

c:\Users\CSIS-PostGrad\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


✅ Batch 1: Downloaded=3157, Skipped=16, Failed=1827, Time=629.05s


train 5000-10000:   0%|          | 0/5000 [00:00<?, ?it/s]

...

test 50000-55000:   0%|          | 0/5000 [00:00<?, ?it/s]

c:\Users\CSIS-PostGrad\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


✅ Batch 11: Downloaded=36729, Skipped=147, Failed=18124, Time=1106.45s


test 55000-59319:   0%|          | 0/4319 [00:00<?, ?it/s]

✅ Batch 12: Downloaded=39625, Skipped=156, Failed=19538, Time=943.82s
💾 Checkpoint + Log saved at 21:52:26
✅ Final checkpoint saved: C:/Users/CSIS-PostGrad/multimodal_pipeline/Data2\checkpoints\test_checkpoint.csv

✅ Done with test: Downloaded=39625, Skipped=156, Failed=19538

🔁 Retry 1 for 19538 failed items...
🔁 Resuming from checkpoint: C:/Users/CSIS-PostGrad/multimodal_pipeline/Data2\checkpoints\test_checkpoint.csv
✅ Final checkpoint saved: C:/Users/CSIS-PostGrad/multimodal_pipeline/Data2\checkpoints\test_checkpoint.csv
❌ Failed list saved: failed_test.csv


In [ ]:
import os
import glob
import time
import requests
import pandas as pd
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display, Image as IPImage, HTML

# ======================================================
# CONFIGURATION
# ======================================================
DATA_DIR = "D:/multimodal_pipeline/data"
TSV_FILES = {
    "train": os.path.join(DATA_DIR, "multimodal_train.tsv"),
    "validate": os.path.join(DATA_DIR, "multimodal_validate.tsv"),
    "test": os.path.join(DATA_DIR, "multimodal_test_public.tsv")
}
OUTPUT_DIRS = {
    "train": os.path.join(DATA_DIR, "train_images"),
    "validate": os.path.join(DATA_DIR, "validate_images"),
    "test": os.path.join(DATA_DIR, "test_images")
}
FAILED_LOG_DIR = os.path.join(DATA_DIR, "failed_downloads")
os.makedirs(FAILED_LOG_DIR, exist_ok=True)
for key, path in OUTPUT_DIRS.items():
    os.makedirs(path, exist_ok=True)

MAX_THREADS = 8
MAX_RETRIES = 3
VISUAL_SAMPLE_EVERY = 100

# ======================================================
# INTERNET CHECK
# ======================================================
_last_check_time = 0
_last_status = True

def internet_on(timeout=5):
    test_urls = ["https://www.google.com", "https://pypi.org", "https://huggingface.co"]
    for url in test_urls:
        try:
            response = requests.get(url, timeout=timeout)
            if response.status_code == 200:
                return True
        except requests.ConnectionError:
            continue
        except Exception:
            continue
    return False

def internet_on_cached(timeout=5, check_interval=15):
    global _last_check_time, _last_status
    now = time.time()
    if now - _last_check_time < check_interval:
        return _last_status
    _last_status = internet_on(timeout)
    _last_check_time = now
    return _last_status

# ======================================================
# VISUALIZATION (OPTIONAL)
# ======================================================
def show_image_sample(image_path, text):
    try:
        display(HTML(f"<b>{text}</b>"))
        display(IPImage(filename=image_path))
    except Exception:
        pass

# ======================================================
# IMAGE DOWNLOAD
# ======================================================
def download_image(row, output_dir, index=None):
    img_id = str(row["id"])
    url = row["image_url"]
    save_path = os.path.join(output_dir, f"{img_id}.jpg")

    if os.path.exists(save_path) or not isinstance(url, str) or not url.startswith("http"):
        return "skipped", None

    while not internet_on_cached():
        print("⚠️ No internet connection. Retrying in 10 seconds...")
        time.sleep(10)

    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            img = Image.open(BytesIO(response.content))
            if img.mode == "P":
                img = img.convert("RGBA")
            else:
                img = img.convert("RGB")
            img.save(save_path)

            if index is not None and index % VISUAL_SAMPLE_EVERY == 0:
                show_image_sample(save_path, row.get("title", "No title"))

            return "downloaded", None
        else:
            return "failed", row
    except Exception:
        return "failed", row

# ======================================================
# PARALLEL DOWNLOAD
# ======================================================
def download_images_parallel(df, output_dir):
    failed_records = []
    skipped_count = downloaded_count = failed_count = 0

    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        futures = {executor.submit(download_image, row, output_dir, idx): idx for idx, row in df.iterrows()}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading images"):
            status, failed_row = future.result()
            if status == "skipped":
                skipped_count += 1
            elif status == "downloaded":
                downloaded_count += 1
            elif status == "failed":
                failed_count += 1
                failed_records.append(failed_row)

    return downloaded_count, skipped_count, failed_records

# ======================================================
# RETRY FAILED DOWNLOADS
# ======================================================
def retry_failed_downloads(failed_records, output_dir):
    for attempt in range(1, MAX_RETRIES + 1):
        if not failed_records:
            break
        print(f"\n🔁 Retry attempt {attempt} for {len(failed_records)} failed downloads...")
        df_failed = pd.DataFrame(failed_records)
        _, _, failed_records = download_images_parallel(df_failed, output_dir)
    return failed_records

# ======================================================
# MAIN FUNCTION WITH SAFE TSV READER
# ======================================================
def download_split(split):
    print(f"\n📦 Loading {split} split...")

    # Try safe reading
    try:
        df = pd.read_csv(
            TSV_FILES[split],
            sep="\t",
            usecols=["id", "image_url", "title"],
            engine="python",
            on_bad_lines="skip"
        )
    except Exception as e:
        print(f"⚠️ Failed to read TSV for {split}: {e}")
        # fallback: try without usecols
        df = pd.read_csv(TSV_FILES[split], sep="\t", engine="python", on_bad_lines="skip")
        print("⚙️ Columns loaded:", df.columns.tolist())

    output_dir = OUTPUT_DIRS[split]
    print(f"Processing {len(df):,} records from '{split}'...")

    # Fast resume check
    existing_files = glob.glob(os.path.join(output_dir, "*.jpg"))
    existing_images = set(os.path.splitext(os.path.basename(f))[0] for f in existing_files)
    print(f"🗂️ Found {len(existing_images):,} already downloaded images.")

    df["id"] = df["id"].astype(str)
    df_to_download = df[~df["id"].isin(existing_images)]
    print(f"🕐 Remaining images to download: {len(df_to_download):,}")

    if df_to_download.empty:
        print(f"✅ All images for '{split}' are already downloaded.")
        return

    downloaded, skipped, failed_records = download_images_parallel(df_to_download, output_dir)
    print(f"\n✅ Pass finished: Downloaded={downloaded}, Skipped={skipped}, Failed={len(failed_records)}")

    failed_records = retry_failed_downloads(failed_records, output_dir)
    print(f"\n❌ Final failed downloads for {split}: {len(failed_records)}")

    if failed_records:
        failed_df = pd.DataFrame(failed_records)
        failed_file = os.path.join(FAILED_LOG_DIR, f"failed_{split}.csv")
        failed_df.to_csv(failed_file, index=False)
        print(f"📝 Logged failed downloads to {failed_file}")

# ======================================================
# EXECUTION
# ======================================================
for split in ["train", "validate", "test"]:
    download_split(split)

print("\n🎉 All splits processed successfully!")


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.